# CNN — Custom Binary Classifier
### *Amphiprion ocellaris* detection | MLEES Master BEC CEE — University of Lausanne

Binary image classifier distinguishing *Amphiprion ocellaris* (clownfish) from morphologically similar marine species, trained on images scraped from the [GBIF API](https://www.gbif.org).

**Pipeline:** data scraping → preprocessing → augmentation → custom CNN → evaluation → XAI saliency maps

---

## 1. Data scraping
Images are fetched from the GBIF occurrence API using `src/data_scraper.py`. The cell below calls the shared module directly.

In [ ]:
import sys
sys.path.insert(0, "src")

from pathlib import Path
from data_scraper import download_images

BASE = Path("dataset")

download_images("Amphiprion ocellaris",        BASE / "Amphiprion_ocellaris_images",    max_images=500)
download_images("Amphiprion clarkii",           BASE / "Amphiprion_clarkii_images",      max_images=300)
download_images("Neoglyphidodon oxyodon",       BASE / "Neoglyphidodon_oxyodon_images",  max_images=300)
download_images("Neopetrolisthes maculatus",    BASE / "Neopetrolisthes_maculatus_images", max_images=300)
download_images("Heteractis aurora",            BASE / "Heteractis_aurora_images",       max_images=300)

## 2. Dataset organisation & splitting
Images are labelled (target / non-target), moved into a structured folder, then split 70 / 15 / 15.

In [ ]:
from dataset import organize_dataset, split_dataset

SOURCE     = Path("dataset")
LABELED    = Path("dataset_labeled")
PROCESSED  = Path("processed")

organize_dataset(SOURCE, LABELED)
split_dataset(LABELED, PROCESSED)

## 3. Preprocessing & augmentation
Images are resized to 224×224, normalised to [0, 1]. Training data is augmented ×5 per image.

In [ ]:
import numpy as np
from pathlib import Path
from dataset import load_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm import tqdm

PROCESSED = Path("processed")

train_data, train_labels = load_split(PROCESSED / "train")
val_data,   val_labels   = load_split(PROCESSED / "val")
test_data,  test_labels  = load_split(PROCESSED / "test")

print(f"Train: {train_data.shape} | Val: {val_data.shape} | Test: {test_data.shape}")

In [ ]:
augmentor = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

aug_data, aug_labels = [], []
for img, label in tqdm(zip(train_data, train_labels), desc="Augmenting", total=len(train_data)):
    flow = augmentor.flow(img[np.newaxis], batch_size=1)
    for _ in range(5):
        aug_data.append(next(flow)[0])
        aug_labels.append(label)

train_data   = np.concatenate([train_data,   np.array(aug_data)])
train_labels = np.concatenate([train_labels, np.array(aug_labels)])

print(f"Augmented train: {train_data.shape}")

## 4. Model architecture
Three convolutional blocks (32 → 64 → 128 filters) with L2 regularisation, 50 % dropout, binary sigmoid output.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2

L2 = 0.01

model = tf.keras.Sequential([
    layers.Conv2D(32,  (3, 3), activation="relu", input_shape=(224, 224, 3), kernel_regularizer=l2(L2)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64,  (3, 3), activation="relu", kernel_regularizer=l2(L2)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation="relu", kernel_regularizer=l2(L2)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu", kernel_regularizer=l2(L2)),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()],
)

model.summary()

## 5. Training
Class weights correct for label imbalance. Early stopping restores best weights; LR is halved on plateau.

In [ ]:
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

LOG_DIR = Path("logs/cnn")
LOG_DIR.mkdir(parents=True, exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)

class_weights = dict(enumerate(
    compute_class_weight("balanced", classes=np.unique(train_labels), y=train_labels)
))

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    tf.keras.callbacks.TensorBoard(log_dir=str(LOG_DIR)),
    tf.keras.callbacks.ModelCheckpoint("checkpoints/cnn_best.keras", save_best_only=True, monitor="val_loss"),
]

history = model.fit(
    train_data, train_labels,
    epochs=30,
    batch_size=32,
    validation_data=(val_data, val_labels),
    class_weight=class_weights,
    callbacks=callbacks,
)

## 6. Evaluation
Loss / accuracy curves, ROC-AUC, confusion matrix and full classification report.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"],     label="train")
ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history.history["accuracy"],     label="train")
ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
from evaluate import evaluate

evaluate(
    model=model,
    test_data=test_data,
    test_labels=test_labels,
    class_names=["non-target", "target"],
    output_dir=Path("results/cnn"),
)

## 7. Explainable AI — saliency maps
Gradient-based saliency maps highlight which pixels drove each prediction.

In [ ]:
from evaluate import plot_saliency_grid

plot_saliency_grid(
    model=model,
    images=test_data,
    output_path=Path("results/cnn/saliency_grid.png"),
    n=4,
)

### Saliency on external images
Drop your own images into `data/external/` to run the model on unseen photos.

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image
from evaluate import plot_saliency_grid

external_dir = Path("data/external")
image_paths  = list(external_dir.glob("*.jpg")) + list(external_dir.glob("*.png"))

external_images = []
for p in image_paths:
    img = Image.open(p).resize((224, 224)).convert("RGB")
    external_images.append(np.array(img, dtype="float32") / 255.0)

if external_images:
    plot_saliency_grid(
        model=model,
        images=np.array(external_images),
        output_path=Path("results/cnn/saliency_external.png"),
        n=len(external_images),
    )
else:
    print("No images found in data/external/ — add .jpg or .png files there.")